# HITO 2 - Analisis de Datos y Formulacion del Problema
## Proyecto MIMIC-TRIAGE (adaptado a PhysioNet/CinC 2012)

Indice:
1) Carga del dataset RAW
2) Inspeccion estructural (IDs, unidades, ventana 0-48h)
3) Distribuciones, cobertura temporal y missingness
4) Correlaciones
5) Analisis por tipo de UCI
6) Scoring probabilistico y ranking clinico
7) Metricas de ranking clinico (Recall@k, % fallecidos en top-k)
8) Metricas probabilisticas (AUC, Brier)
Conclusiones


## 0. Importacion de librerias


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path(".").resolve()
DATASET_ROOT = ROOT / "predicting-mortality-of-icu-patients-the-physionet-computing-in-cardiology-challenge-2012-1.0.0"
SET_A_DIR = DATASET_ROOT / "set-a"
OUTCOMES_A_PATH = DATASET_ROOT / "Outcomes-a.txt"
CSVS_DIR = ROOT / "csvs"
FIGURES_DIR = ROOT / "reports" / "figures"

CSVS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def parse_time_to_hours(time_series: pd.Series) -> pd.Series:
    parts = time_series.astype(str).str.split(":", expand=True)
    return parts[0].astype(int) + (parts[1].astype(int) / 60.0)


def read_set_file(file_path: Path) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    required_cols = {"Time", "Parameter", "Value"}
    missing_cols = required_cols.difference(df.columns)
    if missing_cols:
        raise ValueError(f"{file_path} missing columns: {sorted(missing_cols)}")

    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    df = df.dropna(subset=["Value"]).copy()
    df["Hours"] = parse_time_to_hours(df["Time"])
    df["RecordID"] = int(file_path.stem)
    return df[["RecordID", "Time", "Hours", "Parameter", "Value"]]


def validate_existing_csvs(csv_dir: Path) -> pd.DataFrame:
    suspicious_cols = {"Length_of_stay", "Survival", "SAPS-I", "SOFA"}
    ignored_files = {
        "processed_features_48h_setA.csv",
        "csv_validation_report.csv",
        "hito2_data_audit_setA.json",
    }

    rows = []
    for csv_file in sorted(csv_dir.glob("*.csv")):
        if csv_file.name in ignored_files:
            continue

        try:
            df = pd.read_csv(csv_file)
            columns = list(df.columns)
            row = {
                "file": csv_file.name,
                "n_rows": int(len(df)),
                "n_cols": int(len(columns)),
                "has_recordid": "RecordID" in columns,
                "has_label_in_hospital_death": "In-hospital_death" in columns,
                "contains_potential_leakage_fields": any(col in suspicious_cols for col in columns),
                "notes": "",
            }

            if row["n_rows"] != 4000:
                row["notes"] = "No parece tabla por estancia de Set A (4000 filas)."
            elif not row["has_recordid"]:
                row["notes"] = "No incluye RecordID."
            elif row["contains_potential_leakage_fields"]:
                row["notes"] = "Incluye variables potencialmente post-ingreso/outcome (revisar leakage)."

            rows.append(row)

        except Exception as exc:
            rows.append(
                {
                    "file": csv_file.name,
                    "n_rows": -1,
                    "n_cols": -1,
                    "has_recordid": False,
                    "has_label_in_hospital_death": False,
                    "contains_potential_leakage_fields": False,
                    "notes": f"No se pudo leer: {exc}",
                }
            )

    report = pd.DataFrame(rows)
    report.to_csv(csv_dir / "csv_validation_report.csv", index=False)
    return report


def build_processed_features(raw_48h: pd.DataFrame, outcomes: pd.DataFrame) -> pd.DataFrame:
    raw_48h = raw_48h.sort_values(["RecordID", "Parameter", "Hours"]).copy()
    grouped = raw_48h.groupby(["RecordID", "Parameter"])["Value"]

    features_long = grouped.agg(
        first="first",
        last="last",
        mean="mean",
        min="min",
        max="max",
        std=lambda x: float(np.std(x.to_numpy(), ddof=0)),
        n_mediciones="count",
    ).reset_index()
    features_long["flag_medido"] = 1

    metrics = [
        "first",
        "last",
        "mean",
        "min",
        "max",
        "std",
        "n_mediciones",
        "flag_medido",
    ]

    wide_parts = []
    for metric in metrics:
        pivot_metric = features_long.pivot(
            index="RecordID",
            columns="Parameter",
            values=metric,
        )
        pivot_metric.columns = [f"{col}_{metric}" for col in pivot_metric.columns]
        wide_parts.append(pivot_metric)

    feature_matrix = pd.concat(wide_parts, axis=1).reset_index()

    n_cols = [c for c in feature_matrix.columns if c.endswith("_n_mediciones")]
    flag_cols = [c for c in feature_matrix.columns if c.endswith("_flag_medido")]
    for col in n_cols + flag_cols:
        feature_matrix[col] = feature_matrix[col].fillna(0).astype(int)

    processed = outcomes[["RecordID", "In-hospital_death"]].merge(
        feature_matrix, on="RecordID", how="left"
    )
    processed = processed.sort_values("RecordID").reset_index(drop=True)
    return processed


def export_figures(raw_48h: pd.DataFrame, processed: pd.DataFrame, outcomes: pd.DataFrame, figures_dir: Path) -> None:
    plt.style.use("ggplot")

    flag_cols = [c for c in processed.columns if c.endswith("_flag_medido")]
    missing_df = (
        pd.DataFrame(
            {
                "Parameter": [c.replace("_flag_medido", "") for c in flag_cols],
                "Presencia_pct": [processed[c].mean() * 100 for c in flag_cols],
            }
        )
        .sort_values("Presencia_pct")
        .reset_index(drop=True)
    )
    missing_df["Missing_pct"] = 100 - missing_df["Presencia_pct"]

    fig, ax = plt.subplots(figsize=(11, 10))
    ax.barh(missing_df["Parameter"], missing_df["Missing_pct"], color="#4c78a8")
    ax.set_title("Missingness por variable (Set A, ventana 0-48h)")
    ax.set_xlabel("% sin medicion en 0-48h")
    ax.set_ylabel("Variable")
    fig.tight_layout()
    fig.savefig(figures_dir / "missingness_setA_0_48h.png", dpi=160)
    plt.close(fig)

    hourly_counts = (
        raw_48h.assign(hour_bin=raw_48h["Hours"].astype(int))
        .groupby("hour_bin")
        .size()
        .reindex(range(49), fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(hourly_counts.index, hourly_counts.values, color="#f58518", linewidth=2)
    ax.set_title("Cobertura temporal de mediciones (Set A, 0-48h)")
    ax.set_xlabel("Hora desde ingreso")
    ax.set_ylabel("Numero de mediciones")
    fig.tight_layout()
    fig.savefig(figures_dir / "temporal_coverage_setA_0_48h.png", dpi=160)
    plt.close(fig)

    volume_by_param = raw_48h.groupby("Parameter").size().sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(volume_by_param.index, volume_by_param.values, color="#54a24b")
    ax.set_title("Top 20 variables por volumen de mediciones (0-48h)")
    ax.set_xlabel("Variable")
    ax.set_ylabel("Numero de mediciones")
    ax.tick_params(axis="x", rotation=75)
    fig.tight_layout()
    fig.savefig(figures_dir / "measurement_volume_top20_setA_0_48h.png", dpi=160)
    plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    age_col = "Age_first"
    if age_col in processed.columns:
        valid_age = processed[age_col].where((processed[age_col] >= 0) & (processed[age_col] <= 120))
        axes[0].hist(valid_age.dropna(), bins=20, color="#4c78a8")
    axes[0].set_title("Edad (Age)")
    axes[0].set_xlabel("Anios")

    gender_col = "Gender_first"
    if gender_col in processed.columns:
        gender_counts = processed[gender_col].fillna(-1).value_counts().sort_index()
        axes[1].bar(gender_counts.index.astype(str), gender_counts.values, color="#e45756")
    axes[1].set_title("Genero")
    axes[1].set_xlabel("Codigo")

    icu_col = "ICUType_first"
    if icu_col in processed.columns:
        icu_counts = processed[icu_col].fillna(-1).value_counts().sort_index()
        axes[2].bar(icu_counts.index.astype(str), icu_counts.values, color="#72b7b2")
    axes[2].set_title("Tipo de UCI")
    axes[2].set_xlabel("ICUType")

    fig.tight_layout()
    fig.savefig(figures_dir / "static_distributions_setA.png", dpi=160)
    plt.close(fig)

    label_counts = outcomes["In-hospital_death"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(["0 (vive)", "1 (fallece)"], label_counts.values, color=["#54a24b", "#e45756"])
    total = label_counts.sum()
    for bar, val in zip(bars, label_counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (total * 0.01),
            f"{val} ({(100 * val / total):.1f}%)",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_title("Desbalance de etiqueta en Set A")
    ax.set_ylabel("Numero de estancias")
    fig.tight_layout()
    fig.savefig(figures_dir / "label_imbalance_setA.png", dpi=160)
    plt.close(fig)

    if "HR_mean" in processed.columns:
        hr0 = processed.loc[processed["In-hospital_death"] == 0, "HR_mean"].dropna()
        hr1 = processed.loc[processed["In-hospital_death"] == 1, "HR_mean"].dropna()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.boxplot([hr0, hr1], tick_labels=["0 (vive)", "1 (fallece)"])
        ax.set_title("HR_mean por etiqueta")
        ax.set_ylabel("HR media en 0-48h")
        fig.tight_layout()
        fig.savefig(figures_dir / "hr_mean_by_outcome_setA.png", dpi=160)
        plt.close(fig)


print("Librerias, rutas y funciones auxiliares listas (todo definido dentro del notebook).")


### Resumen de la seccion anterior
- Se importaron librerias de analisis, visualizacion y manejo de rutas para ejecutar el pipeline de extremo a extremo.
- Se fijaron semillas y directorios de trabajo para garantizar reproducibilidad en carga, procesamiento y export de resultados.
- Se definieron en el propio notebook las funciones auxiliares de lectura, validacion, construccion de features y generacion de figuras.


## 1) Carga del Dataset (RAW CinC 2012)


In [ ]:
print("Repo scan breve (archivos clave):")
key_items = [
    ROOT / "AGENT_CONTEXT.md",
    ROOT / "HARP_Hito2.ipynb",
    ROOT / "DataPrepPhysionetChallenge" / "Physionet_2012.ipynb",
    ROOT / "DataPrepPhysionetChallenge" / "Data_Prep_Pipe.ipynb",
    ROOT / "csvs",
    SET_A_DIR,
    DATASET_ROOT / "Outcomes-a.txt",
]
for item in key_items:
    print(f"- {item.relative_to(ROOT)} | existe={item.exists()}")

set_a_files = sorted(SET_A_DIR.glob("*.txt"), key=lambda p: int(p.stem))
print(f"Numero de estancias en set-a: {len(set_a_files)}")
print(f"Primeros RecordID en set-a: {[int(p.stem) for p in set_a_files[:5]]}")


### Resumen de la seccion anterior
- Se localizaron y verificaron los activos clave del repo para Hito 2 (contexto, plantilla, referencias, datos RAW y carpeta de CSVs).
- Se confirmo que `set-a` contiene estancias individuales en formato largo (`Time,Parameter,Value`) y se listaron IDs iniciales para validacion de carga.
- Se establecio la base de trabajo: combinar RAW de `set-a` con `Outcomes-a.txt` para construir una tabla por estancia.


## 2) Inspeccion estructural


In [ ]:
sample_raw = pd.read_csv(set_a_files[0])
outcomes_a = pd.read_csv(OUTCOMES_A_PATH)

print("Muestra RAW (archivo de una estancia):")
display(sample_raw.head(12))

print("Muestra Outcomes-a:")
display(outcomes_a.head(10))

raw_frames = [read_set_file(path) for path in set_a_files]
raw_all = pd.concat(raw_frames, ignore_index=True)
raw_48h = raw_all.loc[raw_all["Hours"] <= 48].copy()

file_ids = pd.Index([int(p.stem) for p in set_a_files], name="RecordID")
outcome_ids = pd.Index(outcomes_a["RecordID"].astype(int), name="RecordID")

audit = pd.Series(
    {
        "set-a_files": len(set_a_files),
        "outcomes_a_rows": len(outcomes_a),
        "ids_set_a_not_in_outcomes": len(file_ids.difference(outcome_ids)),
        "ids_outcomes_not_in_set_a": len(outcome_ids.difference(file_ids)),
        "rows_raw_total": len(raw_all),
        "rows_raw_0_48h": len(raw_48h),
        "max_hour_raw": float(raw_all["Hours"].max()),
        "n_unique_parameters": raw_all["Parameter"].nunique(),
    }
)
display(audit.to_frame("valor"))


### Resumen de la seccion anterior
- Se comprobo la estructura del dataset con evidencia: un archivo por estancia (`RecordID`) y etiqueta en `Outcomes-a.txt`.
- Se verifico correspondencia 1:1 entre estancias de `set-a` y filas de outcomes para Set A.
- Se aplico y audito la ventana temporal `<=48h` como restriccion explicita de construccion de features.


In [ ]:
static_params = ["RecordID", "Age", "Gender", "Height", "ICUType", "Weight"]
param_stays = (
    raw_48h.groupby("Parameter")["RecordID"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_estancias")
)
param_table = param_stays.to_frame().reset_index().rename(columns={"index": "Parameter"})
param_table["presencia_pct"] = 100 * param_table["n_estancias"] / len(set_a_files)

units_map = {
    "Age": "years",
    "Weight": "kg",
    "Height": "cm",
    "HR": "bpm",
    "RespRate": "breaths/min",
    "Temp": "C",
    "SysABP": "mmHg",
    "DiasABP": "mmHg",
    "MAP": "mmHg",
    "NISysABP": "mmHg",
    "NIDiasABP": "mmHg",
    "NIMAP": "mmHg",
    "PaO2": "mmHg",
    "PaCO2": "mmHg",
    "FiO2": "fraction",
    "Urine": "mL",
    "WBC": "K/uL",
    "Glucose": "mg/dL",
    "Creatinine": "mg/dL",
}
param_table["unidad_referencia"] = param_table["Parameter"].map(units_map).fillna("ver docs challenge")

print("Variables detectadas (top 20 por presencia en estancias):")
display(param_table.head(20))
print(f"Variables estaticas esperadas: {static_params}")


### Resumen de la seccion anterior
- Se inspecciono la cobertura de variables por estancia y se construyo una tabla de presencia para priorizar variables con mayor soporte de datos.
- Se anadieron unidades de referencia para las variables clinicas principales, facilitando interpretacion y control de calidad.
- Este bloque deja definida la estructura de variables estaticas y temporales que alimenta el procesamiento posterior.


## 3) Distribuciones


In [ ]:
csv_validation = validate_existing_csvs(CSVS_DIR)
print("Validacion de CSVs existentes en ./csvs (antes de reutilizar):")
display(csv_validation)

processed_48h = build_processed_features(raw_48h=raw_48h, outcomes=outcomes_a)
processed_path = CSVS_DIR / "processed_features_48h_setA.csv"
processed_48h.to_csv(processed_path, index=False)

export_figures(raw_48h=raw_48h, processed=processed_48h, outcomes=outcomes_a, figures_dir=FIGURES_DIR)

print(f"Dataset procesado guardado en: {processed_path}")
print(f"Forma final: {processed_48h.shape}")
print(f"Tasa de mortalidad Set A: {100 * processed_48h['In-hospital_death'].mean():.2f}%")


### Resumen de la seccion anterior
- Se validaron los CSVs existentes en `./csvs/` antes de reutilizarlos como entrada de modelado.
- Se detecto que los CSVs heredados son tablas resumen y no una matriz por estancia 0-48h, ademas de posibles campos con riesgo de leakage.
- Por trazabilidad y coherencia metodologica, el dataset para Hito 3 se reconstruyo desde RAW y se guardo en `processed_features_48h_setA.csv`.


## 3.1.) Distribuciones (II): estaticas, cobertura temporal y missingness


In [ ]:
flag_cols = [c for c in processed_48h.columns if c.endswith("_flag_medido")]
missingness = pd.DataFrame(
    {
        "Parameter": [c.replace("_flag_medido", "") for c in flag_cols],
        "presencia_pct": [100 * processed_48h[c].mean() for c in flag_cols],
    }
).sort_values("presencia_pct")
missingness["missing_pct"] = 100 - missingness["presencia_pct"]

display(missingness.head(15))
display(missingness.tail(15))

figures_created = sorted([p.name for p in FIGURES_DIR.glob("*.png")])
print("Figuras exportadas en reports/figures:")
for f in figures_created:
    print("-", f)


### Resumen de la seccion anterior
- Se cuantifico missingness por variable a partir de `flag_medido`, permitiendo identificar variables con alta y baja cobertura.
- Se verifico el conjunto de figuras exportadas para documentar cobertura temporal, distribuciones estaticas y desbalance de etiqueta.
- Este resumen conecta la exploracion descriptiva con decisiones de seleccion y tratamiento de variables para modelado.


## 4) Correlaciones


In [ ]:
mean_cols = [c for c in processed_48h.columns if c.endswith("_mean")]
selected = [c for c in ["HR_mean", "MAP_mean", "SysABP_mean", "RespRate_mean", "Temp_mean", "Creatinine_mean", "WBC_mean"] if c in mean_cols]

if len(selected) >= 2:
    corr = processed_48h[selected].corr()
    fig, ax = plt.subplots(figsize=(7, 5))
    img = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(selected)))
    ax.set_yticks(range(len(selected)))
    ax.set_xticklabels(selected, rotation=45, ha="right")
    ax.set_yticklabels(selected)
    ax.set_title("Correlaciones (subset de variables medias 0-48h)")
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    corr_path = FIGURES_DIR / "correlation_subset_setA_0_48h.png"
    fig.savefig(corr_path, dpi=160)
    plt.close(fig)
    display(corr.round(2))
    print(f"Figura guardada: {corr_path}")
else:
    print("No hay suficientes columnas para matriz de correlacion.")


### Resumen de la seccion anterior
- Se calcularon correlaciones en un subconjunto de variables medias 0-48h para detectar relaciones lineales relevantes.
- Se genero y guardo una figura de correlacion para soporte del informe y trazabilidad del analisis.
- El resultado orienta controles de colinealidad y la eleccion de baselines lineales/no lineales en fases posteriores.


## 5) Analisis por tipo de UCI
Adaptacion del bloque "analisis por posicion" de la plantilla: aqui segmentamos por `ICUType` para evaluar heterogeneidad clinica.


In [ ]:
if "ICUType_first" in processed_48h.columns:
    cohort = (
        processed_48h.groupby("ICUType_first", dropna=False)
        .agg(
            n_estancias=("RecordID", "count"),
            mortalidad_pct=("In-hospital_death", lambda s: 100 * s.mean()),
            edad_media=("Age_first", "mean"),
            hr_media=("HR_mean", "mean"),
        )
        .sort_values("n_estancias", ascending=False)
    )
    display(cohort.round(2))
else:
    print("ICUType_first no encontrado.")


### Resumen de la seccion anterior
- Se segmentaron las estancias por `ICUType` para comparar volumen, mortalidad y perfiles clinicos agregados.
- El analisis evidencia heterogeneidad entre subcohortes, util para interpretar diferencias de riesgo base.
- Esta vista estratificada ayuda a definir validaciones y analisis de fairness/robustez en Hito 3.


## 6) Scoring (probabilidad de mortalidad => ranking)


In [ ]:
candidate_features = [
    "Age_first",
    "Gender_first",
    "ICUType_first",
    "GCS_mean",
    "HR_mean",
    "MAP_mean",
    "SysABP_mean",
    "DiasABP_mean",
    "RespRate_mean",
    "Temp_mean",
    "Creatinine_mean",
    "BUN_mean",
    "WBC_mean",
    "Urine_mean",
]
features = [f for f in candidate_features if f in processed_48h.columns]
print(f"Features disponibles para baseline: {features}")

model_df = processed_48h[["RecordID", "In-hospital_death"] + features].copy()
for c in features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

is_test = (model_df["RecordID"] % 5 == 0)
train_df = model_df.loc[~is_test].copy()
test_df = model_df.loc[is_test].copy()

medians = train_df[features].median()
X_train = train_df[features].fillna(medians).to_numpy(dtype=float)
X_test = test_df[features].fillna(medians).to_numpy(dtype=float)
y_train = train_df["In-hospital_death"].to_numpy(dtype=float)
y_test = test_df["In-hospital_death"].to_numpy(dtype=int)

mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
sigma[sigma == 0] = 1.0
X_train = (X_train - mu) / sigma
X_test = (X_test - mu) / sigma

def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))

def fit_logreg_gd(X, y, lr=0.05, n_iter=4000, l2=1e-4):
    n, m = X.shape
    w = np.zeros(m)
    b = 0.0
    for _ in range(n_iter):
        p = sigmoid(X @ w + b)
        err = p - y
        grad_w = (X.T @ err) / n + l2 * w
        grad_b = err.mean()
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b

w, b = fit_logreg_gd(X_train, y_train)
test_proba = sigmoid(X_test @ w + b)

ranking_test = test_df[["RecordID", "In-hospital_death"]].copy()
ranking_test["pred_proba_mortality"] = test_proba
ranking_test = ranking_test.sort_values("pred_proba_mortality", ascending=False).reset_index(drop=True)
display(ranking_test.head(20))


### Resumen de la seccion anterior
- Se entreno un baseline de scoring probabilistico con variables agregadas 0-48h y particion reproducible por `RecordID`.
- Se estimaron probabilidades de mortalidad por estancia y se construyo el ranking descendente de prioridad clinica.
- Este bloque formaliza la transicion de datos procesados a decision operativa basada en riesgo.


## 7) MRR (adaptado): metricas de ranking clinico


In [ ]:
def ranking_metrics(y_true, y_score, ks=(25, 50, 100, 200)):
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    total_pos = max(y_true.sum(), 1)
    rows = []
    for k in ks:
        k = min(k, len(y_sorted))
        top_k = y_sorted[:k]
        deaths_top_k = int(top_k.sum())
        rows.append(
            {
                "k": k,
                "deaths_top_k": deaths_top_k,
                "Recall_at_k": deaths_top_k / total_pos,
                "pct_fallecidos_en_top_k": deaths_top_k / k,
            }
        )
    return pd.DataFrame(rows)

rank_metrics = ranking_metrics(y_true=y_test, y_score=test_proba, ks=(25, 50, 100, 200))
display(rank_metrics)


### Resumen de la seccion anterior
- Se calcularon metricas top-k orientadas a priorizacion (Recall@k y porcentaje de fallecidos en top-k).
- El analisis mide de forma directa cuanto riesgo real concentra el tramo alto del ranking.
- Estas metricas complementan las probabilisticas para evaluar utilidad clinica en escenarios de recursos limitados.


## 8) MAP (adaptado): metricas probabilisticas (AUC, Brier)


In [ ]:
def auc_roc_rank(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    pos = y_true == 1
    neg = y_true == 0
    n_pos = pos.sum()
    n_neg = neg.sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = pd.Series(y_score).rank(method="average").to_numpy()
    auc = (ranks[pos].sum() - (n_pos * (n_pos + 1) / 2)) / (n_pos * n_neg)
    return float(auc)

auc = auc_roc_rank(y_test, test_proba)
brier = float(np.mean((test_proba - y_test) ** 2))

prob_metrics = pd.DataFrame(
    {
        "metrica": ["AUC_ROC", "Brier_score", "prevalencia_test"],
        "valor": [auc, brier, float(np.mean(y_test))],
    }
)
display(prob_metrics)


### Resumen de la seccion anterior
- Se evaluo discriminacion global con AUC-ROC y calidad probabilistica con Brier score en el conjunto de prueba.
- El bloque cierra la evaluacion tecnica del baseline combinando perspectiva de ranking y calibracion.
- Con estas salidas queda preparado el punto de partida para comparativas de modelos en Hito 3.


# Conclusiones
- La estructura RAW queda documentada con evidencia: estancia por archivo (`RecordID`) y etiqueta en `Outcomes-a.txt`.
- Se construyo un dataset por estancia con agregaciones 0-48h y bandera de medicion por variable, guardado en `csvs/processed_features_48h_setA.csv`.
- Se exportaron figuras clave para informe en `reports/figures/`.
- Se reformulo el problema en terminos de probabilidad de mortalidad y ranking clinico, reportando Recall@k, % fallecidos top-k, AUC y Brier.
